# 🏦 Exercícios SQL — CVM Data Lake
### Versão do Aluno | Big Data for Finance — FAE
**Banco:** PostgreSQL (DBeaver local)

**Tabelas utilizadas:**
- `layer_02_silver` — Tabelas que estão na camada Silver

**CNPJs de referência rápida:**
| Empresa | CNPJ |
|---|---|
| Petrobras | `33.000.167/0001-01` |
| Ambev | `07.526.557/0001-00` |
| Vale | `33.592.510/0001-54` |

---

## ⚙️ Configuração da Conexão

In [ ]:
from dotenv import load_dotenv
from urllib.parse import quote_plus
from sqlalchemy import create_engine, text
import pandas as pd
import os

In [ ]:
load_dotenv('../.env')

# 2. Lê as variáveis de ambiente
DB_USER = os.getenv('DB_USER')
DB_PASS = os.getenv('DB_PASS')
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_NAME')

if DB_PASS == None:
    print("DB_PASS is empty or null")
elif DB_PASS == "":
    print("DB_PASS is empty or null")

# 3. Encoding de segurança: transforma 'senha@123' em 'senha%40123'
encoded_pass = quote_plus(DB_PASS)

# 4. Monta e cria a engine SQLAlchemy
conn_string = f"postgresql+psycopg2://{DB_USER}:{encoded_pass}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
try:
    engine = create_engine(conn_string)
    print(f'Engine Created Succesfully! With the following credentials: \n USER: {DB_USER} \n HOST: {DB_HOST} \n PORT: {DB_PORT} \n NAME: {DB_NAME}')
except:
    print('Engine Creation Failed, please check credentials')

---
## Questão 1 — Primeira olhada na tabela Silver 🟢 Fácil
**Contexto:** A primeira coisa que um analista faz ao chegar em um novo banco é explorar o que existe dentro da tabela. A camada Silver já tem os dados do Balanço Patrimonial limpos, normalizados e prontos para análise.

**Pergunta:** Mostre as **10 primeiras linhas** da tabela `n1_dfp_cia_aberta_bp`, exibindo apenas as colunas: `CNPJ_CIA`, `DENOM_CIA`, `DT_REFER`, `CD_CONTA`, `DS_CONTA`, `VL_CONTA_TRATADO` e `IS_LEAF`.

In [ ]:
with engine.connect() as conn:
    df_question01 = pd.read_sql(
        text("SELECT "CNPJ_CIA", DENOM_CIA, DT_REFER, CD_CONTA, DS_CONTA, VL_CONTA_TRATADO, IS_LEAF FROM layer_02_silver.n1_dfp_cia_aberta_bp LIMIT 10"),
        con=conn
    )

df_question01

---
## Questão 2 — Volume e cobertura da base 🟢 Fácil
**Contexto:** Antes de qualquer análise, o analista precisa entender o tamanho e a cobertura da base: quantas linhas existem, quantas empresas distintas estão cobertas e qual o intervalo de datas disponível.

**Pergunta:** Em uma única query, calcule para a tabela `n1_dfp_cia_aberta_bp`:
- Total de linhas → `total_linhas`
- Quantidade de empresas distintas (por `CNPJ_CIA`) → `empresas_distintas`
- Quantidade de datas de referência distintas (`DT_REFER`) → `datas_distintas`

In [ ]:
with engine.connect() as conn:
    df_question02 = pd.read_sql(
        text("SELECT COUNT(*) AS total_linhas," \
        "COUNT(DISTINCT CNPJ_CIA) AS empresas_distintas," \
        "COUNT(DISTINCT DT_REFER) AS datas_distintas " \
        "FROM layer_02_silver.n1_dfp_cia_aberta_bp"),
        con=conn
    )

df_question02

---
## Questão 3 — Distribuição de empresas por setor 🟢 Fácil
**Contexto:** A CVM classifica cada empresa listada em um setor de atividade (`SETOR_ATIV`). Entender essa distribuição é o ponto de partida para qualquer análise setorial comparativa. Use a tabela `n0_empresas_selecionadas`, que contém o cadastro completo das empresas do nosso universo.

**Pergunta:** Quantas empresas existem em cada `SETOR_ATIV`? Ordene do setor com **mais** empresas para o com **menos**.

In [ ]:
with engine.connect() as conn:
    df_question03 = pd.read_sql(
        text("SELECT SETOR_ATIV," \
        "COUNT(*)," \
        "COUNT(DISTINCT DT_REFER) AS datas_distintas " \
        "FROM layer_02_silver.n0_empresas_selecionadas" \
        "GROUP BY SETOR_ATIV"),
        con=conn
    )

df_question03

---
## Questão 4 — Petrobras: Ativo e Passivo mais recentes 🟢 Fácil
**Contexto:** No Balanço Patrimonial da CVM, `CD_CONTA = '1'` é o **Ativo Total** e `CD_CONTA = '2'` é o **Passivo + Patrimônio Líquido Total**. A equação fundamental do BP é: **Ativo = Passivo + PL** — qualquer diferença indica erro de dados.

**Pergunta:** Para a **Petrobras** (`CNPJ_CIA = '33.000.167/0001-01'`), mostre o valor (em bilhões de reais) das contas `'1'` e `'2'` para a **data de referência mais recente** disponível. Use uma subconsulta escalar com `MAX("DT_REFER")` para encontrar a data mais recente.

*Dica: `ROUND((valor / 1e9)::NUMERIC, 2)` converte para bilhões com 2 casas.*

In [ ]:
with engine.connect() as conn:
    df_question04 = pd.read_sql(
        text("SELECT CNPJ_CIA," \
        "CD_CONTA," \
        "ROUND((VL_CONTA_TRATADO / 1e9)::NUMERIC, 2) AS valor_bilhao" \
        "FROM layer_02_silver.n1_dfp_cia_aberta_bp" \
        "WHERE CNPJ_CIA = '033.000.167/0001-01'" \
        "AND CD_CONTA IN ('1', '2')" \
        "AND DT_REFER = (SELECT MAX('DT_REFER') FROM layer_02_silver.n1_dfp_cia_aberta_bp WHERE CNPJ_CIA = '033.000.167/0001-01')"),
        con=conn
    )

df_question04

---
## Questão 5 — IS_LEAF: folhas vs. agregadores 🟢 Fácil
**Contexto:** `IS_LEAF = TRUE` marca contas **analíticas** (sem filhos) — são as únicas que devem ser somadas em qualquer ferramenta de BI, pois evitam dupla contagem. `IS_LEAF = FALSE` são contas **sintéticas** (pais) que já incorporam os valores de todas as suas contas filhas. Somar tudo geraria dupla contagem.

**Pergunta:** Para a **Petrobras** (`CNPJ_CIA = '33.000.167/0001-01'`) em `DT_REFER = '2023-12-31'`, quantas contas são `IS_LEAF = TRUE` e quantas são `IS_LEAF = FALSE`? Use `GROUP BY "IS_LEAF"`.

In [ ]:
with engine.connect() as conn:
    df_question05 = pd.read_sql(
        text("SELECT IS_LEAF," \
        "COUNT(*)" \
        "FROM layer_02_silver.n1_dfp_cia_aberta_bp" \
        "WHERE CNPJ_CIA = '33.000.167/0001-01'" \
        "AND DT_REFER = '2023-12-31'" \
        "GROUP BY IS_LEAF"),
        con=conn
    )

df_question05

---
## Questão 6 — Top 10 maiores empresas por Ativo Total em 2023 🟡 Médio
**Contexto:** Rankings de tamanho de balanço são análises clássicas de mercado de capitais. O Ativo Total (`CD_CONTA = '1'`) resume o tamanho total de uma empresa — tudo que ela possui e tem direito a receber.

**Pergunta:** Liste as **10 empresas com maior Ativo Total** em `DT_REFER = '2023-12-31'`. Exiba `DENOM_CIA`, `SETOR_ATIV` e o valor em **bilhões** (2 casas decimais). Ordene do maior para o menor.

*Dica: `ROUND((expr)::NUMERIC, 2)` — o cast para `NUMERIC` é obrigatório no PostgreSQL quando a coluna é `DOUBLE PRECISION`.*

In [ ]:
with engine.connect() as conn:
    df_question06 = pd.read_sql(
        text("SELECT DENOM_CIA," \
        "SETOR_ATIV" \
        "ROUND((VL_CONTA_TRATADO / 1e9)::NUMERIC, 2) AS valor_bilhao" \
        "FROM layer_02_silver.n1_dfp_cia_aberta_bp" \
        "WHERE DT_REFER = '2023-12-31'" \
        "AND CD_CONTA = '1'" \
        "ORDER BY valor_bilhao DESC" \
        "LIMIT 10"),
        con=conn
    )

df_question06

---
## Questão 7 — Evolução anual do Ativo Total da Ambev 🟡 Médio
**Contexto:** Acompanhar a evolução do balanço ao longo dos anos revela a trajetória de crescimento (ou retração) de uma empresa. `DT_REFER` é do tipo `TIMESTAMP` — use `EXTRACT(YEAR FROM ...)` para extrair o ano.

**Pergunta:** Para a **Ambev** (`CNPJ_CIA = '07.526.557/0001-00'`), mostre a evolução anual do **Ativo Total** (`CD_CONTA = '1'`). Extraia o ano de `DT_REFER`, exiba o valor em bilhões e ordene do ano mais antigo ao mais recente.

In [ ]:
with engine.connect() as conn:
    df_question07 = pd.read_sql(
        text("SELECT EXTRACT(YEAR FROM DT_REFER) AS ano," \
        "ROUND((VL_CONTA_TRATADO / 1e9)::NUMERIC, 2) AS valor_bilhao" \
        "FROM layer_02_silver.n1_dfp_cia_aberta_bp" \
        "WHERE CNPJ_CIA = '07.526.556/0001-00'" \
        "ORDER BY ano ASC"),
        con=conn
    )

df_question07

---
## Questão 8 — Contas reconstruídas pelo pipeline 🟡 Médio
**Contexto:** `FLAG_RECONSTRUCAO = TRUE` indica que uma linha foi criada **sinteticamente** pelo pipeline — a empresa não reportou aquela conta e o pipeline reconstruiu o valor somando os filhos. Mais contas reconstruídas = mais "buracos" no dado original enviado à CVM.

**Pergunta:** Em **toda a base histórica**, quais são as **10 combinações de empresa e data** com mais contas reconstruídas no BP? Mostre `DT_REFER`, `DENOM_CIA` e o total de contas de cada empresa. Use `SUM(CASE WHEN "FLAG_RECONSTRUCAO" = TRUE THEN 1 ELSE 0 END)` para contar. Agrupe por `"DT_REFER"` e `"DENOM_CIA"`. Ordene da mais para a menos reconstruída.


In [ ]:
with engine.connect() as conn:
    df_question08 = pd.read_sql(
        text("SELECT DT_REFER," \
        "DENOM_CIA," \
        "SUM(CASE WHEN FLAG_RECONSTRUCAO = TRUE THEN 1 ELSE 0 END) AS top_reconstrucao" \
        "FROM layer_02_silver.n1_dfp_cia_aberta_bp" \
        "GROUP BY DT_REFER, DENOM_CIA" \
        "ORDER BY top_reconstrucao DESC" \
        "LIMIT 10"),
        con=conn
    )

df_question08

---
## Questão 9 — Base analítica de Receita Bruta por empresa via JOIN 🟡 Médio
**Contexto:** Antes de agregar, o analista precisa da base analítica: uma linha por empresa com seu setor e receita. Com esse DataFrame em mãos, o Pandas consegue gerar estatísticas descritivas por grupo com `.describe()`.

**Pergunta:** Faça um `JOIN` entre `n1_dfp_cia_aberta_dre` e `n0_empresas_selecionadas`. Para `DT_REFER = '2023-12-31'` e `CD_CONTA = '3.01'` (Receita Bruta), traga uma linha por empresa com `CNPJ_CIA`, `DENOM_CIA`, `SETOR_ATIV` e o valor em bilhões. Depois use `.groupby` + `.describe()` para ver as estatísticas por setor.


In [ ]:
with engine.connect() as conn:
    df_question09 = pd.read_sql(
        text("SELECT CNPJ_CIA," \
        "DENOM_CIA," \
        "SETOR_ATIV," \
        "ROUND((VL_CONTA_TRATADO / 1e9)::NUMERIC, 2) AS valor_bilhao" \
        "FROM layer_02_silver.n1_dfp_cia_aberta_dre" \
        "RIGHT JOIN layer_02_silver.n0_empresas_selecionadas" \
        "ON n1_dfp_cia_aberta_dre.CNPJ_CIA = n0_empresas_selecionadas.CNPJ_CIA" \
        "WHERE DT_REFER = '2023-12-31'" \
        "AND CD_CONTA = '3.01'" \
        "GROUP BY CNPJ_CIA, DENOM_CIA, SETOR_ATIV"),
        con=conn
    )

df_question09.describe()